# 17 Speedup vs resources

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part IV — Working on a cluster</span>
    <span class="bp-meta">Notebook&nbsp;17</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    Turning a timing table into a defensible core count: computing
    <code>speedup</code> and <code>efficiency</code> with <code>awk</code>, reading
    off the knee, and justifying how many cores to request.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v0.1.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root and source the validation gate. The
# scaling dataset under data/ is read-only; the few results we save go to a
# fresh scratch/.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
cd "$ROOT"

## What this notebook is about

In Notebook 16 you wrote `#SBATCH --ntasks=?` and `--cpus-per-task=?` and simply
**guessed** the numbers. This notebook answers the question properly — and it does
so with the one bash skill that runs through all of Part II: **extract a table and
compute on it with `awk`.** The scaling ideas (efficiency, Amdahl, the knee) are
just enough context to read the numbers; this is a shell notebook, not a
parallel-computing course.

The data is a table of `(cores, walltime)` for the **same job** run on different
core counts — physics-optional, as ever. It is a strong-scaling sweep of the
**graphene + nitric-acid (`gr2hno3`) system** whose CP2K logs you grepped in
Notebooks 6–8, run on ETH's **Euler**. From it you will compute speedup and
efficiency, find where adding cores stops paying off, and write the short
data-backed paragraph that justifies an allocation request.

## A. The question: how many cores?

The temptation is to ask for as many cores as possible — surely more is faster? It
is not, and the cost makes the mistake expensive. A cluster bills you in
**core-hours**:

> **core-hours = cores × walltime.**

A job on 256 cores for 1 hour costs 256 core-hours, the same as 1 core for 256
hours. So if doubling the cores does **not** roughly halve the walltime, you are
paying double for little gain — and on a finite allocation, burning hours you will
want later. The whole notebook is about finding the core count where you are still
getting your money's worth, and stopping there.

## B. Strong scaling from data

Here is the sweep — one row per run, the same problem on more and more cores:

In [2]:
grep -v '^#' data/results/gr2hno3-scaling.dat

1        1830


2         921


4         470


8         243


16        129


32         72


64         53


128        41


256        37


Two derived quantities turn this into an answer. **Speedup** is how many times faster
you got relative to one core, **`S(N) = T(1) / T(N)`**; **efficiency** is how much of
each core you are actually using, **`E(N) = S(N) / N`**. `E = 1` (100%) is the ideal,
where N cores give exactly N× the speed. Both are pure arithmetic on the table, so
`awk` computes them in one pass (the `T(1)` baseline is captured from the first row):

In [3]:
awk '
  /^#/ || NF != 2 { next }                 # skip comments and blanks
  NR_data++ == 0  { t1 = $2 }              # first data row is the 1-core baseline
  {
    S = t1 / $2                            # speedup  S(N) = T(1)/T(N)
    E = S / $1                             # efficiency E(N) = S/N
    printf "%6d  %9d  %8.1f  %7.0f%%\n", $1, $2, S, E*100
  }
  BEGIN { printf "%6s  %9s  %8s  %8s\n", "cores", "time_s", "speedup", "efficncy" }
' data/results/gr2hno3-scaling.dat

 cores     time_s   speedup  efficncy


     1       1830       1.0      100%


     2        921       2.0       99%


     4        470       3.9       97%


     8        243       7.5       94%


    16        129      14.2       89%


    32         72      25.4       79%


    64         53      34.5       54%


   128         41      44.6       35%


   256         37      49.5       19%


Read down the efficiency column and the story jumps out: near-perfect to 32 cores,
then a cliff. Here it is as a picture:

<div style="background:#1b2233;border:1px solid #0e1422;border-radius:8px;padding:8px 20px 16px;margin:18px 0;">
  <div style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:.12em;text-transform:uppercase;color:#c0851a;font-weight:600;margin:8px 0 6px;">Strong scaling of the gr2hno3 job on Euler <span style="text-transform:none;">(32 cores / node)</span></div>
  <pre style="background:transparent;color:#cdd2db;margin:0;font-family:'JetBrains Mono',monospace;font-size:0.84rem;line-height:1.5;">  cores   speedup   parallel efficiency  E = S(N)/N
    1       1.0×    ████████████████████  100%
    2       2.0×    ████████████████████   99%
    4       3.9×    ███████████████████    97%
    8       7.5×    ███████████████████    94%
   16      14.2×    ██████████████████     89%
   32      25.4×    ████████████████       79%   ◀ knee — one full node
   64      34.5×    ███████████            54%   ◀ now across 2 nodes: comm cost
  128      44.6×    ███████                35%
  256      49.5×    ████                   19%   communication-bound
          ideal speedup would be S = N (linear); the gap is parallel overhead</pre>
</div>

The **knee** is the useful core count: the largest N where efficiency is still
respectable — convention puts the line around **75–80%**. Past it, you spend cores
to buy almost nothing. Find it directly: the largest N with `E ≥ 0.75`:

In [4]:
awk '
  /^#/ || NF != 2 { next }
  NR_data++ == 0  { t1 = $2 }
  { if ((t1/$2)/$1 >= 0.75) knee = $1 }
  END { print "knee (largest core count with E >= 0.75):", knee, "cores" }
' data/results/gr2hno3-scaling.dat

knee (largest core count with E >= 0.75): 32 cores


**32 cores** — and that is exactly **one node** of this machine. The reason is
physical: up to 32 cores the job lives inside a single node, sharing fast memory;
at 64 it must split across **two** nodes and pay for every byte they exchange over
the network. That communication cost is what bends the curve down so sharply past
the node boundary.

## C. Why it bends: Amdahl's law

The bend is not bad luck; it is arithmetic. Almost every job has a **serial
fraction** `s` — setup, I/O, reductions — that cannot be parallelized. If a fraction
`s` of the work is stuck on one core and the rest `(1 − s)` splits perfectly, then

> **S(N) = 1 / ( s + (1 − s)/N )**   — Amdahl's law.

As `N → ∞`, the `(1 − s)/N` term vanishes and speedup hits a **ceiling** of `1/s`,
no matter how many cores you throw at it. You can estimate `s` straight from the
data: rearranging Amdahl for a single measured point `N` (using the ratio
`r = T(N)/T(1)`) gives `s = (r·N − 1) / (N − 1)`. Take an in-node point, where the
formula is clean:

In [5]:
awk '
  /^#/ || NF != 2 { next }
  NR_data++ == 0  { t1 = $2 }
  $1 == 16 {
    r = $2 / t1
    s = (r*$1 - 1) / ($1 - 1)
    printf "serial fraction s = %.3f  ->  Amdahl ceiling S_max = 1/s = %.0f\n", s, 1/s
  }
' data/results/gr2hno3-scaling.dat

serial fraction s = 0.009  ->  Amdahl ceiling S_max = 1/s = 117


So under one percent of this job is serial — excellent — and Amdahl alone would
allow a speedup near 100×. Yet the data tops out around **50×**: the extra shortfall
is the **inter-node communication** of §B, which Amdahl's single-node model does not
include. (The mirror image is **Gustafson's law / weak scaling**: if you grow the
*problem* with the cores instead of fixing it, efficiency holds up far better — a
different question for a different day.)

## D. Measure your own

The dataset shows the **node-level** rolloff. You can feel a **core-level** one on
this very machine. Take a fixed batch of CPU work, run it split across 1, then 2,
then 4 parallel workers (`xargs -P`, from Notebook 5), and time each with the
`date`/`awk` stopwatch from Notebook 13:

In [6]:
cd "$ROOT"

In [7]:
# A "work unit" is one short, CPU-bound awk loop (no I/O, repeatable). bench runs
# a fixed batch of 4 of them across W parallel workers and returns the wall time.
bench() {
  local W="$1" t0 t1
  t0=$(date +%s.%N)
  seq 1 4 | xargs -P "$W" -I{} bash -c 'awk "BEGIN{s=0;for(i=0;i<4000000;i++)s+=sqrt(i)}"'
  t1=$(date +%s.%N)
  awk "BEGIN{ printf \"%.3f\", $t1 - $t0 }"
}

In [8]:
t1=$(bench 1)
printf "%7s  %7s  %7s\n" workers time_s speedup
for W in 1 2 4; do
  tw=$(bench "$W")
  awk "BEGIN{ printf \"%7d  %7.2f  %7.2f\n\", $W, $tw, $t1/$tw }"
done

workers   time_s  speedup


      1     1.04     1.00


      2     1.07     0.97


      4     1.07     0.97


You should see speedup climb and then **flatten** — going from 2 to 4 workers buys
much less than 1 to 2 — because once you ask for more workers than the machine has
physical cores, they fight over the same hardware (**oversubscription**). That is the
core-level twin of the node-level cliff in §B.

:::{admonition} ⚠ The shape is the lesson, not the numbers
:class: warning
This page runs on a small, **shared** cloud machine (often only 1–2 cores you do not
have to yourself), so your exact times will be noisy and your speedup may stall at 2
workers or wobble. That is fine — even expected. Read the **shape** (more workers →
diminishing returns → a rolloff), not the decimals. Real benchmarking pins a job to
dedicated cores and averages several runs; here we are after the idea.
:::

Put the two together and you have the whole picture: your single-node run shows the
rolloff from **oversubscription** (too many workers per core); the dataset shows the
rolloff from **communication** (too many nodes per job). Both say the same thing —
*there is a point past which more resources stop helping.*

## E. The justification

The payoff is a decision you can defend. An allocation committee (CSCS, LUMI, your PI)
does not want "we'd like a lot of cores"; it wants a number with evidence. The
efficiency table writes the paragraph for you:

> *We request **32 cores** (one node) per job. Strong-scaling tests of this system
> show parallel efficiency holding above **79%** through 32 cores, then falling
> sharply — to **54%** at 64 cores — as the job spans multiple nodes and
> inter-node communication dominates. Running at 32 cores therefore uses our
> allocation efficiently; larger jobs would burn roughly double the core-hours for
> well under double the speed.*

Numbers in, a defensible decision out. That is the entire point of the exercise — and
the reason the shell work mattered: it turned a raw timing table into the one sentence
that gets the allocation approved.

## Exercises

The dataset analysis is read-only and deterministic — graded on the numbers. The live
micro-benchmark's times vary with the machine, so it is graded on **structure** (the
right rows, a rolloff), not exact seconds. Saved results go to a fresh `scratch/`.

### Warm-up 1 (worked) — Speedup and efficiency

Compute the full `S` and `E` table from the dataset with `awk`, and read off the
efficiency at 32 cores.

In [9]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [10]:
# (solution hidden on the public site)


 cores     time_s   speedup    effic.


     1       1830      1.00      100%


     2        921      1.99       99%


     4        470      3.89       97%


     8        243      7.53       94%


    16        129     14.19       89%


    32         72     25.42       79%


    64         53     34.53       54%


   128         41     44.63       35%


   256         37     49.46       19%


In [11]:
e32=$(awk '/^#/||NF!=2{next} NR_data++==0{t1=$2} $1==32{printf "%.0f",(t1/$2)/$1*100}' data/results/gr2hno3-scaling.dat)
s32=$(awk '/^#/||NF!=2{next} NR_data++==0{t1=$2} $1==32{printf "%.1f",t1/$2}' data/results/gr2hno3-scaling.dat)
check '[ "$e32" = "79" ] && [ "$s32" = "25.4" ]' \
      "awk computed the speedup/efficiency table correctly (S=25.4x, E=79% at 32 cores)"

✓ awk computed the speedup/efficiency table correctly (S=25.4x, E=79% at 32 cores)


### Warm-up 2 (your turn) — Find the knee

Identify the largest core count whose efficiency is still at least 75%.

In [12]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [13]:
# (solution hidden on the public site)


knee: 32 cores (efficiency stays >= 75% up to here)


In [14]:
check '[ "$knee" = "32" ]' "the knee — the largest core count with E >= 0.75 — is 32"

✓ the knee — the largest core count with E >= 0.75 — is 32


### Applied 1 (your turn) — Amdahl's serial fraction

Estimate the serial fraction `s` from an in-node data point with
`s = (r·N − 1)/(N − 1)`, `r = T(N)/T(1)`, and report the Amdahl ceiling `1/s`.

In [15]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [16]:
# (solution hidden on the public site)


from N=8:  s = 0.0089  ->  S_max = 1/s = 112


In [17]:
s=$(awk '/^#/||NF!=2{next} NR_data++==0{t1=$2} $1==8{r=$2/t1; printf "%.4f",(r*$1-1)/($1-1)}' data/results/gr2hno3-scaling.dat)
check 'awk "BEGIN{ exit !('"$s"' >= 0.005 && '"$s"' <= 0.02) }"' \
      "the estimated serial fraction s ($s) is small and in the expected range (~0.005-0.02)"

✓ the estimated serial fraction s (0.0089) is small and in the expected range (~0.005-0.02)


### Applied 2 (your turn) — Measure your own speedup

Time a fixed batch of CPU work at 1, 2, and 4 parallel workers (`xargs -P` + the
`date`/`awk` stopwatch), build the speedup table, and note where it rolls off. Times
vary, so only the structure is graded.

In [18]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
bench() { local W="$1" t0 t1; t0=$(date +%s.%N); seq 1 4 | xargs -P "$W" -I{} bash -c 'awk "BEGIN{s=0;for(i=0;i<4000000;i++)s+=sqrt(i)}"'; t1=$(date +%s.%N); awk "BEGIN{printf \"%.3f\", $t1-$t0}"; }

In [19]:
# (solution hidden on the public site)


workers   time_s  speedup


      1     1.04     1.00


      2     1.07     0.97


      4     1.07     0.97


In [20]:
rows=$(grep -cE '^[[:space:]]*[124][[:space:]]' scratch/my-speedup.txt)
check '[ "$rows" = "3" ] && grep -q "speedup" scratch/my-speedup.txt' \
      "the micro-benchmark produced a 3-row speedup table (1, 2, 4 workers)"

✓ the micro-benchmark produced a 3-row speedup table (1, 2, 4 workers)


### Composite — putting it together (justify an allocation)

The real proposal skill: from the dataset, recommend a core count and back it with the
efficiency evidence. Compute the knee, then write a short justification paragraph that
cites the efficiency at the knee and the collapse beyond it. Save it to a file.

In [21]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [22]:
# (solution hidden on the public site)


We request 32 cores (one node) per job. Strong-scaling tests show parallel


efficiency holding at 79% through 32 cores, then falling to 54%


once the job spans multiple nodes and inter-node communication dominates. Running at


32 cores uses the allocation efficiently; larger jobs would cost far more


core-hours for little extra speed.


In [23]:
check '[ "$knee" = "32" ] && grep -q "32 cores" scratch/justification.txt && grep -qiE "efficiency" scratch/justification.txt' \
      "the justification recommends 32 cores and cites the efficiency evidence"

✓ the justification recommends 32 cores and cites the efficiency evidence


### Optional stretch — Plot it in the terminal

No grade. Turn the speedup column into a quick ASCII bar chart with `awk` (no new
tools) — the shell's instant plot:

In [24]:
cd "$ROOT"

In [25]:
# (solution hidden on the public site)


   1 cores  S=  1.0  #


   2 cores  S=  2.0  ##


   4 cores  S=  3.9  ####


   8 cores  S=  7.5  ########


  16 cores  S= 14.2  ##############


  32 cores  S= 25.4  #########################


  64 cores  S= 34.5  ###################################


 128 cores  S= 44.6  #############################################


 256 cores  S= 49.5  #################################################


For a real plot, pipe the two columns to **`gnuplot`** in your terminal
(`gnuplot -p -e "plot 'data/results/gr2hno3-scaling.dat' using 1:2 with linespoints"`)
— or load them into Python/matplotlib. Same data, prettier axes.

## Outlook

You can now turn a timing table into a defensible core count — speedup and efficiency
with `awk`, the knee, the Amdahl ceiling, and the paragraph that justifies the request.
That is the last piece of **working on a cluster**, and it closes Part IV.

What is left is to stop doing all of this **by hand.** You have a scaling sweep
(Notebook 16), text extraction (Part II), and the analysis above — separate steps you
run and re-run manually. **Part V** ties the whole course together with **`make`**
(Notebook 18): a single `make` that orchestrates the sweep, the parsing, and the
analysis as one automated, repeatable pipeline — scripting, text processing, and HPC
awareness composed into the tool that runs them all.

_No new commands this time: this is a pure-analysis notebook, and every tool in it —
`awk` (Notebook 8), `sort`, `xargs -P` (Notebook 5), `time`/`date` (Notebook 13) —
you have already met. The Compendium is unchanged._

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box to run the
    <code>awk</code> analysis and your own micro-benchmark, and to try the
    <code>gnuplot</code> plot. The published notebooks ship <b>without worked
    solutions</b>; if you would like the reference solutions — to teach from or to
    check your own work — get in touch:
    <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>